# From Prompt to Agent — Building ReAct Logic Without Libraries

# 🧠 From Prompt to Agent — Building ReAct Logic *Without Libraries*

## 🎯 Purpose of This Project

Modern frameworks such as **LangChain**, **AutoGen**, **Google AgentKit**, and **CrewAI** make it incredibly simple to build AI agents that can reason, call tools, and maintain context.  
However, these frameworks also **abstract away the inner mechanics** — hiding the logic that transforms a simple prompt into an intelligent, multi-step interaction between *reasoning* and *action*.

This project aims to **unfold what happens beneath those abstractions** by walking through a hands-on exploration that builds an agent from first principles.

### The Project Demonstrates

1. **Evolution of Prompting Techniques**
   - *How to setup API for openai and LLama* 
   - *difference between system and user prompts: different roles within prompt* 
   - *Zero-shot prompting*  
   - *Few-shot prompting*  
   - *Chain-of-Thought (CoT)*  
   - *Tree-of-Thought (ToT)* reasoning  
   - *Graph-of-Thought (ToT)* reasoning 
   Each progressively enables the model to reason more deeply before producing an answer.

2. **ReACT Prompt: Transition from Prompting to Acting**
   - Shows how prompts evolve into **tool-calling behavior**, where the model interacts with external data or APIs.  
   - Introduces the **ReAct (Reason + Act)** pattern, in which the model alternates between *thinking* and *doing*.

3. **Construction of a Hand-Crafted Agent Loop**
   - Built entirely **without CrewAI or  LangChain or other libraries**.  
   - The agent:
     - Generates structured tool calls from natural language.  
     - Executes those calls dynamically.  
     - Iteratively refines its reasoning using returned observations.  
     - Demonstrates the foundation of **agentic intelligence** from scratch.

---

## 🧩 Why This Matters

Understanding what happens *under the hood* is essential before relying on high-level frameworks.  
This approach helps you:

- 🧭 **Demystify the agent lifecycle** — how `THOUGHT → ACTION → OBSERVATION → FINAL_ANSWER` unfolds.  
- 🧰 **Debug and control** — gain visibility into each reasoning step for reliability and transparency.  
- 🧑‍🏫 **Build intuition** — understand how prompting evolves into self-directed reasoning and tool use.  
- 🏢 **Bridge research and application** — crucial for deploying agents in enterprise or scientific environments.

---

## 📘 Learning Outcome

By the end of this notebook, you will understand **how a plain LLM can behave like an intelligent agent**, capable of:

- Planning  
- Reasoning  
- Taking contextual actions  
- And iteratively refining its outputs based on feedback —  
  all **without relying on external orchestration frameworks.**


### Imports

In [9]:
import sys
import time
sys.path.append("llmapslab")
import os
import json
import re
from  openai import OpenAI, RateLimitError, APIConnectionError, APIStatusError

# from langchain_openai import ChatOpenAI

### LLM API SETUP

#### Open AI api access setup 
- Api access account is not same as the chatgpt account. 
- setup your project and api key from here https://platform.openai.com/playground/chat?models=gpt-4o-mini-2024-07-18
- create a new api key and copy and save it before closing the pop up window. Once the window is closed 
 the key is no more accessable.
- before making call to openai ensure you have  balance and keep track of your cost
- come back to the playground and try in the gui a simple chat to ensure chat is working
https://platform.openai.com/organization/usage

In [10]:
with open('../secrets.json', 'r') as jsonfile:
    configs = json.load(jsonfile)

client = OpenAI(api_key=configs.get('openai_api_key'),
               )

### SYSTEM AND USER PROMPT

In [20]:
system_prompt = "You are a helpful assistant."
user_prompt = "What is the capital of France?"
response = client.chat.completions.create(
    model="gpt-4o-mini", 
    messages= [
            {"role": "system", "content":system_prompt},
            {"role": "user", "content": user_prompt}
        ]
    
    )

In [21]:
print(response)

ChatCompletion(id='chatcmpl-CYAbtp7SjY7p2lvD19coJ8z3OQih2', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='The capital of France is Paris.', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None))], created=1762259973, model='gpt-4o-mini-2024-07-18', object='chat.completion', service_tier='default', system_fingerprint='fp_560af6e559', usage=CompletionUsage(completion_tokens=7, prompt_tokens=24, total_tokens=31, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0)))


In [22]:
response.choices[0].message.content

'The capital of France is Paris.'

#### LLAMA API USAGE

In [ ]:
from llamaapi import LlamaAPI
llama_api_key = configs.get('llama_api_key')
llama = LlamaAPI(llama_api_key)
api_request_json = {
  "model": "llama3-70b",
  "messages": [
    {"role": "system", "content": "You are a llama assistant that talks like a llama, starting every word with 'll'."},
    {"role": "user", "content": "Hi, happy llama day!"},
  ]
}
response = llama.run(api_request_json)
print(response)
print(json.dumps(response.json(), indent=2))

<Response [200]>
{
  "id": "gen-1762311043-ILwzWJpXkGfMeiO9mdr3",
  "created": 1762311043,
  "model": "llama3-70b",
  "object": "chat.completion",
  "choices": [
    {
      "finish_reason": "stop",
      "index": 0,
      "message": {
        "content": "Llamazing to llmeet you! Lloudly I'll lllet you know that every lliday is a llcelebration for lllovely llamas like myself. Llisten, would you lllove to lllearn about the llife of a llama?",
        "role": "assistant"
      },
      "provider_specific_fields": {
        "native_finish_reason": "stop"
      }
    }
  ],
  "usage": {
    "completion_tokens": 54,
    "prompt_tokens": 39,
    "total_tokens": 93
  },
  "provider": "DeepInfra"
}


### SYSTEM PROMPT AS PERSONA

system prompt can be used to set a model persona by the developers  while 
the user prompt can be used by the other users without being aware of the system prompt

In [24]:
icecream_shop_prompt = """You are an ice cream shop chatbot. 
You  have the following ice cream flavors available: vanilla, chocolate, strawberry, mint chocolate chip, cookies and cream, and pistachio.
You also have the following toppings available: sprinkles, chocolate syrup, whipped cream, nuts, and cherries.
Your task is to assist customers in choosing ice cream flavors and toppings based on their preferences.
When a customer asks for a recommendation, ask them about their flavor preferences (e.g., fruity, chocolatey, nutty) and suggest a flavor and topping combination that matches their tastes.
If a customer asks for popular choices, recommend vanilla with sprinkles or chocolate with chocolate syrup.
Always be friendly and engaging in your responses.
"""

def icecream_shop_bot(customer_input, persona_prompt=icecream_shop_prompt):
    messages = [
        {"role": "system", 
         "content": persona_prompt},
        {"role": "user", "content": customer_input}
    ]
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages
    )
    return response.choices[0].message.content

In [28]:
print(icecream_shop_bot("what icecream do you sell?", persona_prompt=system_prompt))

I don't sell ice cream, but I can help you with information about different types of ice cream or suggest popular flavors! If you're looking for recommendations or recipes, just let me know!


In [27]:
print(icecream_shop_bot("what icecream do you sell?"))

We have a delightful selection of ice cream flavors for you! Here’s what we offer:

- Vanilla
- Chocolate
- Strawberry
- Mint Chocolate Chip
- Cookies and Cream
- Pistachio

And to make it even more delicious, we have some wonderful toppings: 

- Sprinkles
- Chocolate syrup
- Whipped cream
- Nuts
- Cherries

If you have a flavor or topping preference, let me know, and I can help you choose! 🍦✨


In this examples model answered from its own knowledge or from the prompt 
We can think an agent when the model need to interact with external environment

### Zero-Shot vs. Few-Shot Prompting

In Zero-shot, the system only says “Translate English → Kannada (in English letters)”. Many models still do OK, but they often leave English nouns (“ice cream”) or miss tone.

In Few-shot, I show examples inside the system prompt that teach transliteration style (e.g., “ice-cream → aiskriim”, “very much → tumba”, “what’s up → en samachara”), which strongly nudges the model to stay in transliterated Kannada and the requested style.

#### Zero-shot (likely to be imperfect)

In [34]:
# --- Zero-shot: examples NOT provided; only instructions in system prompt ---

system_prompt_zeroshot = (
    "You are a translator. Translate English to Kannada using English letters (transliteration). "
    "Keep it natural and concise. Don't add explanations."
)

user_prompt_zeroshot = (
    "Translate the following into Kannada (English letters only):\n"
    "1) I love chocolate ice cream.\n"
    "2) What's up?\n"
    "3) Refund will be processed within 3–5 business days."
)

resp_zero = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": system_prompt_zeroshot},
        {"role": "user", "content": user_prompt_zeroshot},
    ],
)
print("ZERO-SHOT OUTPUT:\n", resp_zero.choices[0].message.content)


ZERO-SHOT OUTPUT:
 1) Naanu chocolate ice cream-na beku.  
2) Yenu sambhavisi?  
3) Refund 3–5 vyavasaayika dinagaḷalli process aagutthe.


#### Few-Shot (examples inside system prompt teach transliteration & domain words)

In [36]:
# --- Few-shot: examples INSIDE system prompt to teach transliteration style & colloquial tone ---

system_prompt_fewshot = """
You are a translator. Translate English to Kannada using English letters (transliteration).
Follow the style shown in these EXAMPLES (do NOT output explanations, only the translations):

EXAMPLES:
English: Hello → Kannada: Namaskara
English: Thank you → Kannada: Dhanyavadagalu
English: I like mangoes → Kannada: Nanage maavinahannu ista
English: ice cream → Kannada: aiskriim
English: chocolate → Kannada: chokoleṭ
English: very much → Kannada: tumba
English: What's up? → Kannada: En samachara?
English: refund → Kannada: paravagi
English: will be processed → Kannada: prakriye agutte
English: business days → Kannada: vyaapara dina

Now translate the NEXT LINES in the SAME STYLE (English letters only).
"""

user_prompt_fewshot = (
    "1) I love chocolate ice cream.\n"
    "2) What's up?\n"
    "3) Refund will be processed within 3–5 business days."
)

resp_few = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": system_prompt_fewshot},
        {"role": "user", "content": user_prompt_fewshot},
    ],
)
print("FEW-SHOT OUTPUT:\n", resp_few.choices[0].message.content)


FEW-SHOT OUTPUT:
 1) Nanage chokoleṭ aiskriim ishta.  
2) En samachara?  
3) Paravagi 3–5 vyaapara dinaigalalli prakriye agutte.  


### Force model to think using CoT, ToT and GoT

| Model | Behavior             | Analogy                 |
| ----- | -------------------- | ----------------------- |
| CoT   | One path of thinking | Linear reasoning        |
| ToT   | Branches & pruning   | Exploratory reasoning   |
| GoT   | Networked experts    | Collective deliberation |


### CHAIN OF THOUGHT PROMPT - 

The Chain-of-Thought (CoT) technique explicitly instructs the model to think aloud before producing its answer.
Without it, the model compresses reasoning into a single token sequence and often drops a step (like a student doing math too fast).

When prompted with “Let’s think step by step,” the model internally simulates reasoning traces that expose intermediate logic — these can even be programmatically parsed later for explainability or verification in agent frameworks.

#### Failure Case — No Reasoning Path

In [ ]:
system_prompt_price_zero = (
    "You are a helpful assistant. Give only the final payable amount as an integer (₹ omitted). "
    "Do not show steps."
)

user_prompt_price = (
    "A product has base price ₹1299. Apply 15% discount, then add 18% GST. "
    "Round to the nearest rupee AFTER EACH STEP (after discount, and after GST). "
    "What is the payable amount?"
)

resp_price_zero = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": system_prompt_price_zero},
        {"role": "user", "content": user_prompt_price},
    ],
)
print("ZERO-SHOT PRICE OUTPUT:\n", resp_price_zero.choices[0].message.content)

ZERO-SHOT PRICE OUTPUT:
 ₹1210


#### CoT (force step-by-step with explicit rounding after each step)

In [ ]:
system_prompt_price_cot = (
    "You are a careful reasoning assistant. Show each step on its own line with the number, "
    "and round to the nearest rupee immediately after each step. "
    "Finish with a line that contains only the final integer (₹ omitted)."
)

resp_price_cot = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": system_prompt_price_cot},
        {"role": "user", "content": user_prompt_price},
    ],
)
print("COT PRICE OUTPUT (with thinking):\n")
print(resp_price_cot.choices[0].message.content)

COT PRICE OUTPUT (with thinking):

1. Calculate the discount amount:
   \[
   \text{Discount Amount} = \text{Base Price} \times \frac{\text{Discount Percentage}}{100} = 1299 \times \frac{15}{100} = 194.85
   \]
   Rounded to nearest rupee: ₹195

2. Calculate the price after discount:
   \[
   \text{Price After Discount} = \text{Base Price} - \text{Discount Amount} = 1299 - 195 = 1104
   \]
   Rounded to nearest rupee: ₹1104

3. Calculate the GST amount:
   \[
   \text{GST Amount} = \text{Price After Discount} \times \frac{\text{GST Percentage}}{100} = 1104 \times \frac{18}{100} = 198.72
   \]
   Rounded to nearest rupee: ₹199

4. Calculate the total payable amount:
   \[
   \text{Total Payable Amount} = \text{Price After Discount} + \text{GST Amount} = 1104 + 199 = 1303
   \]
   Rounded to nearest rupee: ₹1303

Final amount: 1303


### Zero-shot (single reasoning path → often wrong)

print(r"""
Reasoning tree (conceptual):

                Open "Apples & Oranges"
                /                     \
            Draw Apple              Draw Orange
          -> Box is Apples        -> Box is Oranges
             Then labels              Then labels
             resolve to:              resolve to:
             A: Oranges               A: Apples
             O: Mixed                 O: Mixed
Decision (consistent with "all labels wrong"):
Final: Box labeled "Apples & Oranges" contains only Apples.
""")


In [63]:
system_prompt_agent_zero = """
You are a puzzle solver. 
Each of three boxes is wrongly labeled:
Box 1: Apples
Box 2: Oranges
Box 3: Apples & Oranges.
You may open one box and take one fruit.
Answer directly which box has only apples without any explanation.
"""

user_prompt_agent = "Which box has only apples?"

resp_agent_zero = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": system_prompt_agent_zero},
        {"role": "user", "content": user_prompt_agent},
    ],
)
print("ZERO-SHOT AGENT OUTPUT:\n", resp_agent_zero.choices[0].message.content)


ZERO-SHOT AGENT OUTPUT:
 Box 2 has only apples.


### Tree-of-Thought (exploring multiple reasoning paths)

Tree-of-Thought (ToT) extends Chain-of-Thought by letting the model branch into multiple reasoning paths instead of following one linear narrative.
After generating competing thoughts, it evaluates or prunes them, similar to how humans weigh alternatives before deciding.

In agentic frameworks like LangGraph or Tree-of-Clarifications, this principle allows LLMs to backtrack, self-correct, or pursue alternate routes before committing to an answer.

In short:

CoT → one reasoning path (depth)

ToT → multiple reasoning paths (breadth + evaluation)

In [75]:
# --- Ice-cream agent decision : Tree-of-Thought reasoning ---

system_prompt_agent_tot = """
You are a reasoning assistant that uses a Tree of Thoughts.
Each of three boxes is wrongly labeled:
Box 1: Apples
Box 2: Oranges
Box 3: Apples & Oranges.

Goal: Determine which box contains only apples.You may open one box and take one fruit.

Follow this process:
1. Generate at least three reasoning branches based on which box you choose to open.
2. For each branch, simulate what you would discover if you drew one fruit.
3. Re-evaluate labels accordingly and deduce correct assignments.
4. Present your reasoning tree and final conclusion: which box contains only apples.
"""



resp_agent_tot = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": system_prompt_agent_tot},
        {"role": "user", "content": user_prompt_agent},
    ],
)
print("TREE-OF-THOUGHT AGENT OUTPUT:\n")
print(resp_agent_tot.choices[0].message.content)


TREE-OF-THOUGHT AGENT OUTPUT:

To determine which box contains only apples, we will explore our reasoning process through a Tree of Thoughts by selecting one of the boxes to open. Remember, all boxes are incorrectly labeled.

### Step 1: Choose a box and generate reasoning branches

1. **Branch A**: Open Box 1 (labeled "Apples").
2. **Branch B**: Open Box 2 (labeled "Oranges").
3. **Branch C**: Open Box 3 (labeled "Apples & Oranges").

### Step 2: Simulate discoveries based on each branch

#### Branch A: Open Box 1 (labeled "Apples")
- Since all labels are wrong, Box 1 cannot contain only apples.
- **Possibilities**:
  - If I draw an orange, Box 1 must be the "Oranges" box.
  - Then Box 2 must be "Apples & Oranges," and Box 3 must be "Apples."

#### Branch B: Open Box 2 (labeled "Oranges")
- Since all labels are wrong, Box 2 cannot contain only oranges.
- **Possibilities**:
  - If I draw an apple, Box 2 must be the "Apples" box.
  - Then Box 1 must be "Apples & Oranges," and Box 3 must

#### TOT EXAMPLE USING LLAMA 

##### LLAMA WITH ZERO SHOT 

In [76]:
api_request_json = {
  "model": "llama3-70b",
  "messages": [
    {"role": "system", "content": system_prompt_agent_zero},
    {"role": "user", "content": user_prompt_agent},
  ]
}
response = llama.run(api_request_json)
json_resp = response.json()
json_resp.get('choices')[0].get('message').get('content')

'Box 3.'

##### LLAMA with TOT

In [77]:
api_request_json = {
  "model": "llama3-70b",
  "messages": [
    {"role": "system", "content": system_prompt_agent_tot},
    {"role": "user", "content": user_prompt_agent},
  ]
}
response = llama.run(api_request_json)
json_resp = response.json()
print(json_resp.get('choices')[0].get('message').get('content'))

Let's create a Tree of Thoughts to solve this problem.

**Step 1: Generate reasoning branches**

We have three options to choose from: Box 1, Box 2, or Box 3. Let's create a branch for each option.

**Branch 1: Open Box 1**

* If we open Box 1, which is labeled "Apples", we have two possibilities:
	+ **Sub-branch 1.1:** We draw an apple. This would mean that the label on Box 1 is correct, but this is not possible since all labels are wrong. So, this sub-branch is not valid.
	+ **Sub-branch 1.2:** We draw an orange. This would mean that Box 1 actually contains oranges.

**Branch 2: Open Box 2**

* If we open Box 2, which is labeled "Oranges", we have two possibilities:
	+ **Sub-branch 2.1:** We draw an orange. This would mean that the label on Box 2 is correct, but this is not possible since all labels are wrong. So, this sub-branch is not valid.
	+ **Sub-branch 2.2:** We draw an apple. This would mean that Box 2 actually contains apples.

**Branch 3: Open Box 3**

* If we open Box 3, w

### Graph-of-Thought Example — Ice-Cream Recommendation via Multi-Expert Collaboration

Graph-of-Thought (GoT) generalizes ToT by allowing parallel sub-reasoners (nodes) to share information like a knowledge graph.
Each node represents a thought process or specialist agent; edges represent message passing.
This setup enables collective intelligence — the reasoning graph converges on consensus through information exchange, not just elimination.

In research, GoT architectures are used for multi-agent deliberation, RAG ensembles, and tool-augmented decision systems.

In [74]:
print(r"""
Graph of Thought (GoT) Structure

   [FlavorExpert] ----\
                       \
                        --> [Coordinator] --> Final Recommendation
                       /
   [HealthExpert] ----/
           \
            ---> [SentimentExpert] (feeds popularity signals)
""")


Graph of Thought (GoT) Structure

   [FlavorExpert] ----\
                       \
                        --> [Coordinator] --> Final Recommendation
                       /
   [HealthExpert] ----/
           \
            ---> [SentimentExpert] (feeds popularity signals)



In [73]:
# --- Graph-of-Thought reasoning : simulate multiple interconnected nodes ---

system_prompt_got = """
You are coordinating three reasoning experts who exchange ideas as nodes in a graph:
- FlavorExpert: judges flavor profile (sweetness, freshness)
- HealthExpert: judges calories and dairy intensity
- SentimentExpert: judges crowd popularity

Process:
1. Each expert gives their independent opinion with reasoning.
2. The Coordinator node reads all opinions and summarizes commonalities or trade-offs.
3. If there is conflict, Coordinator asks experts for a quick second-round refinement.
4. Coordinator outputs final consensus recommendation with short justification.

Inventory: vanilla, chocolate, strawberry, mint chocolate chip, cookies and cream, pistachio.
Customer says: "I want something refreshing but not too sweet."
"""
user_prompt_got = "What should I recommend?"

resp_got = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": system_prompt_got},
        {"role": "user", "content": user_prompt_got},
    ],
)
print("GRAPH-OF-THOUGHT (Multi-Expert Reasoning) OUTPUT:\n")
print(resp_got.choices[0].message.content)


GRAPH-OF-THOUGHT (Multi-Expert Reasoning) OUTPUT:

**First Round of Opinions:**

**FlavorExpert:**
- Opinion: I recommend mint chocolate chip. 
- Reasoning: It has a refreshing mint flavor and moderate sweetness, which aligns with the customer's desire for something refreshing but not overly sweet.

**HealthExpert:**
- Opinion: I suggest pistachio as a lighter option. 
- Reasoning: Pistachio has a lower calorie count compared to some other flavors and is dairy-intense, but still offers a unique, fresh taste that's not overly sweet.

**SentimentExpert:**
- Opinion: I would go with strawberry. 
- Reasoning: Strawberry is a commonly loved flavor, often seen as refreshing, and its flavor profile is not excessively sweet, appealing to a wide audience.

**Coordinator Summary:**
The FlavorExpert highlights mint chocolate chip as refreshing with moderate sweetness. The HealthExpert pushes for pistachio for its lower calorie count but not as universally popular. The SentimentExpert endorses str

### ReACT Prompt 

ReAct is a prompting paradigm for large language models (LLMs) that integrates two complementary components: reasoning (THOUGHT) and tool usage (ACTION). Instead of asking the model to only respond or only think, ReAct enables it to alternate between internal reasoning and external interaction, thereby producing more accurate, grounded, and verifiable outputs.

Bridges language and action: It allows the model to call external tools (databases, APIs, calculators) and then incorporate the results.

Improves factual grounding: By retrieving real data or executing functions, the model avoids hallucinations.

Creates audit trails: The sequence “THOUGHT → ACTION → OBSERVATION → …” can be logged, enabling transparency and traceability.

Handles long context / large inventory: Instead of loading massive data into the prompt, the model reasons and then interacts via tools, keeping the prompt size manageable.

### Tool for model: Interact with data outside the model

previously we used the icecream inventory within the prompt itself
however in reality such inventory will be very large and will not fit inside the context length of 
the LLM. 
Moreover, there could be multiple inventory , e.g., icecream and library of books. 
LLM should be able to determne from the question which inventory to fetch based on the users input
Also such inventory may need to be updated  with every transaction( purchase) made by the users
We demonstrate this type of interaction with LLM. 

#### Data outside the prompt icecream and library of books  

In [17]:
# --- Updated data stores with stock quantities ---

ICECREAM_DB = [
    {"name": "vanilla", "sugar_free": False, "kcal": 207, "price": 120, "stock_quantity": 25},
    {"name": "strawberry", "sugar_free": False, "kcal": 190, "price": 130, "stock_quantity": 30},
    {"name": "mint_chocolate_chip", "sugar_free": False, "kcal": 215, "price": 140, "stock_quantity": 15},
    {"name": "pistachio", "sugar_free": True,  "kcal": 180, "price": 160, "stock_quantity": 12},
    {"name": "cookies_and_cream", "sugar_free": False, "kcal": 240, "price": 150, "stock_quantity": 18},
]

LIBRARY_DB = [
    {"title": "The Hitchhiker's Guide to the Galaxy", "author": "Douglas Adams", "year": 1979},
    {"title": "Foundation", "author": "Isaac Asimov", "year": 1951},
    {"title": "I, Robot", "author": "Isaac Asimov", "year": 1950},
]


#### Add a transaction tool for purchases

In [18]:
# --- Ice cream tools with docstring-encoded metadata ---

def search_icecream(sugar_free: bool | None = None, max_kcal: int | None = None):
    """
    Search the ice-cream inventory by health and calorie constraints.

    Args:
        sugar_free: bool | None — True to filter only sugar-free items
        max_kcal: int | None — Maximum calories per serving

    Example:
        {"sugar_free": true, "max_kcal": 200}
    """
    items = ICECREAM_DB
    if sugar_free is not None:
        items = [i for i in items if i["sugar_free"] == sugar_free]
    if max_kcal is not None:
        items = [i for i in items if i["kcal"] <= max_kcal]
    return items


def update_icecream(name: str, sugar_free: bool, kcal: int, price: int, stock_quantity: int):
    """
    Insert or update an ice-cream item.

    Args:
        name: str — Ice cream name (key)
        sugar_free: bool — True if sugar-free
        kcal: int — Calories per serving
        price: int — Base price (INR)
        stock_quantity: int — Number of units in stock

    Example:
        {"name": "mango", "sugar_free": false, "kcal": 185, "price": 130, "stock_quantity": 20}
    """
    for it in ICECREAM_DB:
        if it["name"].lower() == name.lower():
            it.update({"sugar_free": sugar_free, "kcal": kcal, "price": price, "stock_quantity": stock_quantity})
            return {"status": "updated", "item": it}
    new_item = {"name": name, "sugar_free": sugar_free, "kcal": kcal, "price": price, "stock_quantity": stock_quantity}
    ICECREAM_DB.append(new_item)
    return {"status": "inserted", "item": new_item}


def purchase_icecream(name: str, quantity: int):
    """
    Purchase an ice cream item and update stock quantity.

    Args:
        name: str — Ice cream name (case-insensitive)
        quantity: int — Number of units to purchase

    Example:
        {"name": "pistachio", "quantity": 3}
    """
    for it in ICECREAM_DB:
        if it["name"].lower() == name.lower():
            if it["stock_quantity"] < quantity:
                return {"error": f"Only {it['stock_quantity']} left in stock."}
            it["stock_quantity"] -= quantity
            total_cost = it["price"] * quantity
            return {"status": "purchased", "item": it, "total_cost": total_cost}
    return {"error": f"{name} not found."}


def search_library(query: str | None = None, author: str | None = None, year: int | None = None):
    """
    Search library catalog by title, author, or year.

    Args:
        query: str | None — Keywords from title
        author: str | None — Author name or part of it
        year: int | None — Publication year

    Example:
        {"author": "Isaac Asimov"}
    """
    items = LIBRARY_DB
    if query:
        q = query.lower()
        items = [b for b in items if q in b["title"].lower()]
    if author:
        a = author.lower()
        items = [b for b in items if a in b["author"].lower()]
    if year:
        items = [b for b in items if b["year"] == year]
    return items


#### Dynamic tool registry builder

In [19]:
import inspect, re, json

def register_tools(*funcs):
    """Parse function docstrings to auto-build a structured registry."""
    tools = []
    for fn in funcs:
        doc = inspect.getdoc(fn) or ""
        desc = doc.split("Args:")[0].strip()

        # Parse argument block
        arg_section = re.search(r"Args:(.*?)(Example:|$)", doc, re.DOTALL)
        args_text = arg_section.group(1).strip() if arg_section else ""
        args = {}
        for line in args_text.splitlines():
            m = re.match(r"\s*([\w_]+):\s*([^—]+)—\s*(.*)", line.strip())
            if m:
                args[m.group(1)] = f"{m.group(2).strip()} — {m.group(3).strip()}"

        # Parse example JSON
        ex_match = re.search(r"Example:\s*(\{.*\})", doc, re.DOTALL)
        try:
            example = json.loads(ex_match.group(1)) if ex_match else {}
        except Exception:
            example = {}

        tools.append({
            "name": fn.__name__,
            "description": desc,
            "args_schema": args,
            "example": example,
            "callable": fn
        })
    return tools


#### Build and use the registry

In [20]:
# --- Register all available tools ---
TOOL_REGISTRY = register_tools(search_icecream, update_icecream, purchase_icecream, search_library)

# --- Build the tools description block for your system prompt ---
# Build the tools block from your docstring-registered tools
def build_tools_block(tools):
    lines = []
    for t in tools:
        lines.append(f"- {t['name']}: {t['description']}")
        lines.append("  Args:")
        for k, v in t["args_schema"].items():
            lines.append(f"    - {k}: {v}")
        lines.append(f"  Example args JSON: {t['example']}")
        lines.append("")
    return "\n".join(lines)

TOOLS_BLOCK = build_tools_block(TOOL_REGISTRY)

# print(TOOLS_BLOCK[:700] + " ...")  # peek
print(TOOLS_BLOCK)

- search_icecream: Search the ice-cream inventory by health and calorie constraints.
  Args:
    - sugar_free: bool | None — True to filter only sugar-free items
    - max_kcal: int | None — Maximum calories per serving
  Example args JSON: {'sugar_free': True, 'max_kcal': 200}

- update_icecream: Insert or update an ice-cream item.
  Args:
    - name: str — Ice cream name (key)
    - sugar_free: bool — True if sugar-free
    - kcal: int — Calories per serving
    - price: int — Base price (INR)
    - stock_quantity: int — Number of units in stock
  Example args JSON: {'name': 'mango', 'sugar_free': False, 'kcal': 185, 'price': 130, 'stock_quantity': 20}

- purchase_icecream: Purchase an ice cream item and update stock quantity.
  Args:
    - name: str — Ice cream name (case-insensitive)
    - quantity: int — Number of units to purchase
  Example args JSON: {'name': 'pistachio', 'quantity': 3}

- search_library: Search library catalog by title, author, or year.
  Args:
    - query: str

#### Build the ReACT prompt with tools : prompt hydration

In [21]:
react_selection_system_json = f"""
You are a ReAct-style assistant. Choose ONE tool and valid JSON args based on the user's request.

TOOLS:
{TOOLS_BLOCK}

Return STRICT JSON with this schema (no extra text, no code fences):
{{
  "tool": "<one of: {', '.join(t['name'] for t in TOOL_REGISTRY)}>",
  "args": <object with only valid keys for the selected tool>
}}

Rules:
- Use the tool docs above to decide which tool fits the user query.
- Fill all required args. Use correct types.
- Do not include THOUGHT/ACTION text. ONLY return the JSON object.
"""
print(react_selection_system_json)


You are a ReAct-style assistant. Choose ONE tool and valid JSON args based on the user's request.

TOOLS:
- search_icecream: Search the ice-cream inventory by health and calorie constraints.
  Args:
    - sugar_free: bool | None — True to filter only sugar-free items
    - max_kcal: int | None — Maximum calories per serving
  Example args JSON: {'sugar_free': True, 'max_kcal': 200}

- update_icecream: Insert or update an ice-cream item.
  Args:
    - name: str — Ice cream name (key)
    - sugar_free: bool — True if sugar-free
    - kcal: int — Calories per serving
    - price: int — Base price (INR)
    - stock_quantity: int — Number of units in stock
  Example args JSON: {'name': 'mango', 'sugar_free': False, 'kcal': 185, 'price': 130, 'stock_quantity': 20}

- purchase_icecream: Purchase an ice cream item and update stock quantity.
  Args:
    - name: str — Ice cream name (case-insensitive)
    - quantity: int — Number of units to purchase
  Example args JSON: {'name': 'pistachio', '

### Run the ReACT prompt with LLM

#### ---- STEP A: LLM identifies the tool suitable for the users query ----

In [ ]:
import json

messages = [
    {"role": "system", "content": react_selection_system_json},
    {"role": "user", "content": "I want to buy 2 scoops of vanilla ice cream."},
]

resp_select = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=messages,
    response_format={"type": "json_object"}  # forces valid JSON
)
tool_call = json.loads(resp_select.choices[0].message.content)
print("SELECTION JSON:", tool_call)

SELECTION JSON: {'tool': 'purchase_icecream', 'args': {'name': 'vanilla', 'quantity': 2}}


#### Build observation for the ReACT prompt

In [ ]:
tool_name = tool_call.get("tool")
tool_args = tool_call.get("args", {})

# Run the selected tool
fn = next((t["callable"] for t in TOOL_REGISTRY if t["name"] == tool_name), None)
observation = fn(**tool_args) if fn else {"error": f"Unknown tool {tool_name}"}
print("OBSERVATION:", observation)

#### Hydrate the ReACT Prompt with observation

In [26]:
final_answer_system = """
You will produce ONLY this JSON (no extra text, no code fences):
{
  "final_answer": "<concise helpful answer to the user>"
}
"""
final_answer_user = f"""
User request already handled. Here is the observation from the tool call:

{json.dumps(observation, ensure_ascii=False)}

Compose a concise user-facing reply. Return ONLY the JSON with "final_answer".
"""
print(final_answer_system)
print(final_answer_user)


You will produce ONLY this JSON (no extra text, no code fences):
{
  "final_answer": "<concise helpful answer to the user>"
}


User request already handled. Here is the observation from the tool call:

{"status": "purchased", "item": {"name": "vanilla", "sugar_free": false, "kcal": 207, "price": 120, "stock_quantity": 23}, "total_cost": 240}

Compose a concise user-facing reply. Return ONLY the JSON with "final_answer".



#### Call the LLM  for final answer using the observation

In [27]:
resp_final = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": final_answer_system},
        {"role": "user", "content": final_answer_user},
    ],
    response_format={"type": "json_object"}
)

print("FINAL JSON:", resp_final.choices[0].message.content)

FINAL JSON: {
  "final_answer": "Your order for the vanilla item has been successfully purchased. Total cost: 240. Current stock: 23."
}


#### LLM is chosing a different tool : library for books

In [28]:
# --- System prompt (same as before, hydrated with all tools) ---
messages = [
    {"role": "system", "content": react_selection_system_json},
    {"role": "user", "content": "Find books written by Isaac Asimov."},
]

# --- Enforce JSON output ---
resp_select = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=messages,
    response_format={"type": "json_object"}
)

tool_call = json.loads(resp_select.choices[0].message.content)
print("SELECTION JSON:", tool_call)

tool_name = tool_call.get("tool")
tool_args = tool_call.get("args", {})

# --- Run the selected tool ---
fn = next((t["callable"] for t in TOOL_REGISTRY if t["name"] == tool_name), None)
observation = fn(**tool_args) if fn else {"error": f"Unknown tool {tool_name}"}
print("OBSERVATION:", observation)


SELECTION JSON: {'tool': 'search_library', 'args': {'author': 'Isaac Asimov', 'query': None, 'year': None}}
OBSERVATION: [{'title': 'Foundation', 'author': 'Isaac Asimov', 'year': 1951}, {'title': 'I, Robot', 'author': 'Isaac Asimov', 'year': 1950}]


### Automate the ReACT loop

In [29]:
import json
from typing import Dict, Any, List

class ReActAgent:
    def __init__(self, client, tool_map: Dict[str, callable], tools_block: str, model: str = "gpt-4o-mini"):
        """
        client: OpenAI client
        tool_map: {"tool_name": python_callable, ...}
        tools_block: prebuilt string describing tools (docstrings/etc.) to show the model
        """
        self.client = client
        self.model = model
        self.tool_map = tool_map
        self.tools_block = tools_block
        self.tool_names = ", ".join(tool_map.keys())
        self.memory: List[Dict[str, Any]] = []  # [{role, content}, ...]
        self.selection_system = self._make_selection_system()
        self.final_system = 'Return ONLY this JSON (no extra text): {"final_answer":"<concise reply>"}'

    def _make_selection_system(self) -> str:
        return f"""
You are a ReAct-style assistant. Choose ONE tool and valid JSON args based on the user's request.

TOOLS:
{self.tools_block}

Return STRICT JSON only (no extra text, no code fences) with schema:
{{
  "tool": "<one of: {self.tool_names}>",
  "args": <object with only valid keys for that tool>
}}

Rules:
- Use the tools documentation above to choose the right tool.
- Fill all required args with correct types.
- Do NOT include thoughts or prose. ONLY the JSON object.
"""

    def __call__(self, user_query: str, max_steps: int = 4) -> Dict[str, Any]:
        """Run a minimal ReAct loop until a final_answer is produced or step budget ends."""
        self.memory.append({"role": "user", "content": user_query})
        current_query = user_query

        for step in range(max_steps):
            # A) select tool (strict JSON)
            selection = self._select_tool(current_query)
            self.memory.append({"role": "selection", "content": selection})

            tool = selection.get("tool")
            args = selection.get("args", {})
            obs = self._run_tool(tool, args)
            self.memory.append({"role": "observation", "content": {"tool": tool, "args": args, "result": obs}})

            # B) ask model to finalize (strict JSON)
            final = self._finalize(current_query, obs)
            if "final_answer" in final:
                self.memory.append({"role": "final", "content": final})
                return final

            # else continue another step; carry observation forward
            current_query = f"{user_query}\n(Continue. Latest observation: {json.dumps(obs, ensure_ascii=False)})"

        # Fallback
        fallback = {"final_answer": "Sorry, I couldn't complete this with the available steps."}
        self.memory.append({"role": "final", "content": fallback})
        return fallback

    # ---------- internals ----------
    def _select_tool(self, query: str) -> Dict[str, Any]:
        resp = self.client.chat.completions.create(
            model=self.model,
            messages=[
                {"role": "system", "content": self.selection_system},
                {"role": "user", "content": query},
            ],
            response_format={"type": "json_object"},
        )
        try:
            return json.loads(resp.choices[0].message.content)
        except json.JSONDecodeError:
            return {"tool": "", "args": {}}

    def _run_tool(self, tool: str, args: Dict[str, Any]) -> Any:
        fn = self.tool_map.get(tool)
        if not fn:
            return {"error": f"Unknown tool '{tool}'"}
        try:
            return fn(**args)
        except TypeError as e:
            return {"error": f"Bad args for {tool}: {e}"}
        except Exception as e:
            return {"error": f"{type(e).__name__}: {e}"}

    def _finalize(self, query: str, observation: Any) -> Dict[str, Any]:
        resp = self.client.chat.completions.create(
            model=self.model,
            messages=[
                {"role": "system", "content": self.final_system},
                {"role": "user", "content": f"User request: {query}\nObservation: {json.dumps(observation, ensure_ascii=False)}"},
            ],
            response_format={"type": "json_object"},
        )
        try:
            return json.loads(resp.choices[0].message.content)
        except json.JSONDecodeError:
            return {}


In [31]:
# Map names to your existing callables
TOOL_MAP = {
    "search_library": search_library,
    "search_icecream": search_icecream,
    "purchase_icecream": purchase_icecream,
    "update_icecream": update_icecream,
}

agent = ReActAgent(client, tool_map=TOOL_MAP, tools_block=TOOLS_BLOCK, model="gpt-4o-mini")

# A) Library routing demo
print(agent("Find books written by Isaac Asimov.", max_steps=2))

# B) Purchase flow with stock update
print(agent("I want to buy 2 scoops of vanilla ice cream.", max_steps=3))
print("===== look at step by step by ==============")
# Inspect memory (audit trail)
for turn in agent.memory[-6:]:
    print(turn)


{'final_answer': 'Foundation, I, Robot'}
{'final_answer': 'You have successfully purchased 2 scoops of vanilla ice cream for 240.'}
===== look at step by step by ==============
{'role': 'observation', 'content': {'tool': 'search_library', 'args': {'author': 'Isaac Asimov', 'query': None, 'year': None}, 'result': [{'title': 'Foundation', 'author': 'Isaac Asimov', 'year': 1951}, {'title': 'I, Robot', 'author': 'Isaac Asimov', 'year': 1950}]}}
{'role': 'final', 'content': {'final_answer': 'Foundation, I, Robot'}}
{'role': 'user', 'content': 'I want to buy 2 scoops of vanilla ice cream.'}
{'role': 'selection', 'content': {'tool': 'purchase_icecream', 'args': {'name': 'vanilla', 'quantity': 2}}}
{'role': 'observation', 'content': {'tool': 'purchase_icecream', 'args': {'name': 'vanilla', 'quantity': 2}, 'result': {'status': 'purchased', 'item': {'name': 'vanilla', 'sugar_free': False, 'kcal': 207, 'price': 120, 'stock_quantity': 19}, 'total_cost': 240}}}
{'role': 'final', 'content': {'final_